# LLMBuilder — Track A training on a free Colab T4 (Phase 3)

**What this does:** fine-tunes the stock Llama 3.2 1B on the Lucid-only corpus
(2,426 rows, Track A — the only track), merges the LoRA adapter into full
weights, and converts the result to **one GGUF** — the snapshot.

**Run top to bottom.** Each step prints what it did; stop at the first error
and fix it before continuing. Nothing here trains on this VM and nothing is
committed to any repo from this notebook.

## Before you start — three things you must have ready

1. **A Colab GPU runtime:** Runtime → Change runtime type → **T4 GPU**.
   The free tier gives ~30 GPU hours/week. This run is ~60–70 min
   (measured 2026-11-10: training alone took 67 min — the fp32 LM head
   on a T4 runs ~26 s/step, not the 10–20 min first estimated).
2. **A Hugging Face account with the Llama license accepted** (the base
   weights are gated). Put a **read-only** Hugging Face access token in a
   Colab secret named **`HF_TOKEN`** (left sidebar → Secrets 🔑 → Add new
   secret, name `HF_TOKEN`, value = the token). **The token is never typed
   into a cell and never printed — the notebook reads it from the secret.**
3. **Three files from the LLMBuilder repo** (tracked since commit
   a7147b2, 2026-11-10 — the clone below brings them; the upload cell
   is now only a fallback if a local checkout is newer):
   - `scripts/training/train_lora.py`
   - `requirements-lora.txt`
   - `configs/training/sft_lora.yaml`

   The upload cell below asks for exactly these three files. Upload them
   from the machine holding the checkout.

## Rules baked in (do not work around them)

- **Training data is ONLY `datasets/lucid_corpus/lucid_corpus.jsonl`.**
  The trainer refuses any path under `datasets/acquired/` (GSM8K is
  evaluation-only, The Stack is held). No external dataset anywhere.
- **Checkpoints go to Google Drive** (`MyDrive/llmbuilder-lora/`) — Colab
  sessions are ephemeral; anything left in `/content` disappears.
- **The GGUF conversion cell uses the llama.cpp converter as a BUILD-TIME
  TOOL ONLY (Gary approved 2026-11-10 16:20 PST).** Gary rejected
  llama.cpp for the inference engine; the converter is a separate
  build-time tool and its use is his call, not assumed here.


In [ ]:
# 0 — Confirm we actually have a GPU (T4 expected)
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU — switch the runtime to T4 GPU first."
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# 1 — Clone the repo (brings the tracked corpus + repo structure).
# NOTE: train_lora.py / requirements-lora.txt / sft_lora.yaml are tracked
# in the repo (commit a7147b2) — the clone brings them. The next cell
# only uploads them if the clone is missing them.
!git clone https://github.com/GareBear99/ARC-Neuron-LLMBuilder.git
%cd /content/ARC-Neuron-LLMBuilder
!git log --oneline -1
!wc -l datasets/lucid_corpus/lucid_corpus.jsonl   # expect 2426


In [ ]:
# 2 — The three files are tracked in the repo (commit a7147b2) and the clone
# above brings them. Only upload if they are missing (e.g. a local checkout
# is newer than the repo). Never upload anything else.
from pathlib import Path
from google.colab import files

NEEDED = {
    "scripts/training/train_lora.py": "train_lora.py",
    "requirements-lora.txt": "requirements-lora.txt",
    "configs/training/sft_lora.yaml": "sft_lora.yaml",
}
missing = [p for p in NEEDED if not Path(p).exists()]
if missing:
    print("Missing from the clone, upload them:", missing)
    uploaded = files.upload()
    for repo_path, up_name in NEEDED.items():
        if up_name in uploaded:
            Path(repo_path).write_bytes(uploaded[up_name])

# Sanity: the trainer must carry the no-external-data refusal.
assert "datasets/acquired" in Path("scripts/training/train_lora.py").read_text()
!ls -l scripts/training/train_lora.py requirements-lora.txt configs/training/sft_lora.yaml
!sha256sum scripts/training/train_lora.py requirements-lora.txt configs/training/sft_lora.yaml


In [ ]:
# 3 — Install the training stack.
# requirements-lora.txt deliberately does NOT list torch: Colab preinstalls
# the CUDA build of torch and installing this file must NOT downgrade it.
!pip install -q -r requirements-lora.txt
import torch, transformers, peft, trl, datasets, accelerate, bitsandbytes
import torchao  # merge needs torchao >= 0.16.0 (0.10.0 broke it 2026-11-10)
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| trl", trl.__version__,
      "| torchao", torchao.__version__)


In [ ]:
# 4 — Mount Google Drive (checkpoints + outputs live here, not /content).
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/llmbuilder-lora
!ls -ld /content/drive/MyDrive/llmbuilder-lora


In [ ]:
# 5 — Hugging Face login from the Colab secret (never typed, never printed).
# Prerequisite: secret HF_TOKEN exists (left sidebar → Secrets) and holds a
# READ-ONLY token from an account that accepted the Meta Llama license.
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError(
        "Colab secret 'HF_TOKEN' is missing. Add it under Secrets (left sidebar), "
        "name exactly HF_TOKEN, value = a read-only HF access token, then re-run this cell.")
login(token=hf_token)
print("HF login OK — token came from the Colab secret, not from any cell.")


## 6 — Dry run first (no weights, no GPU needed)

Validates the config + the 2,426-row corpus and resolves the tokenizer
(downloads the tokenizer only). Expect `{"ok": true, "mode": "dry_run", ...
"rows_usable": 2426, ...}` and a `run_manifest.json` in the output dir.
A non-zero exit + a clear `ERROR:` means fix the setup before training.


In [ ]:
# 6 — Dry run: config + dataset + tokenizer only.
!python scripts/training/train_lora.py --dry-run


## 7 — Train (the real run, ~10–20 min on a free T4)

Exact command, with every Phase 3 setting explicit (the rest are the
trainer's defaults, from `configs/training/sft_lora.yaml`):

- base model: `meta-llama/Llama-3.2-1B-Instruct` (gated — needs the HF login above)
- dataset: `datasets/lucid_corpus/lucid_corpus.jsonl` — 2,426 rows, Lucid-only
- QLoRA 4-bit NF4, LoRA r=64, alpha=128, dropout 0.05
- lr 2e-4, 2 epochs, max_seq 2048, micro-batch 1 × grad-accum 32 (effective batch 32)
- output: the PEFT adapter + tokenizer + `run_manifest.json` on Drive

The run prints `{"ok": true, "mode": "train", ...}` with the final loss and
global step when it finishes.


In [ ]:
# 7 — Full Track A training run.
!python scripts/training/train_lora.py \
  --model meta-llama/Llama-3.2-1B-Instruct \
  --dataset datasets/lucid_corpus/lucid_corpus.jsonl \
  --output-dir /content/drive/MyDrive/llmbuilder-lora/run1


## 8 — Merge the adapter into full weights

The trainer saves a PEFT adapter (`adapter_model.safetensors` +
`adapter_config.json`), not a full model. The merge loads the base model in
bf16 (NOT 4-bit — standard practice for merging a QLoRA adapter) and folds
the adapter in with PEFT's `merge_and_unload()`. Output is a normal
Hugging Face checkpoint on Drive: `run1-merged/`.


In [ ]:
# 8 — Merge LoRA adapter -> full bf16 model.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE = "meta-llama/Llama-3.2-1B-Instruct"
ADAPTER = "/content/drive/MyDrive/llmbuilder-lora/run1"
MERGED = "/content/drive/MyDrive/llmbuilder-lora/run1-merged"

base = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=False)
model = PeftModel.from_pretrained(base, ADAPTER)
merged = model.merge_and_unload()
merged.save_pretrained(MERGED, safe_serialization=True)
tok = AutoTokenizer.from_pretrained(ADAPTER)
tok.save_pretrained(MERGED)
print("merged model saved to", MERGED)

# Stability fixes, applied here so a re-merge stays convertible:
# - the converter asserts max(tokenizer vocab) < vocab_size; the merged
#   config ships 128256 but the tokenizer max id is 128256, so the
#   assert fails. Set vocab_size to 128257 (proven 2026-11-10).
# - the converter needs tokenizer_class LlamaTokenizerFast.
import json
from pathlib import Path as _P
_cfg = _P(MERGED) / "config.json"
_c = json.loads(_cfg.read_text())
_c["vocab_size"] = 128257
_cfg.write_text(json.dumps(_c, indent=2))
_tc = _P(MERGED) / "tokenizer_config.json"
_t = json.loads(_tc.read_text())
_t["tokenizer_class"] = "LlamaTokenizerFast"
_tc.write_text(json.dumps(_t, indent=2))
print("vocab_size set to 128257; tokenizer_class set to LlamaTokenizerFast")


## 9 — Convert the merged model to ONE GGUF

**Converter approved as a build-time tool only** (Gary, 2026-11-10 16:20 PST).
The llama.cpp *converter* (`convert_hf_to_gguf.py`) is approved; the llama.cpp
*inference engine* remains rejected. There is no other converter in the repo.

**Two steps, both on Drive** (never `/content` — a runtime disconnect wipes it;
the F16 built 2026-11-10 was lost exactly that way):
1. Convert to **F16** first — `--outtype q4_k_m` is **invalid** (the converter
   only accepts f32, f16, bf16, q8_0, tq1_0, tq2_0, auto). Default is F16.
2. Build `llama-quantize` and quantize the F16 to **Q4_K_M** (~0.75 GB).

Both files land on Drive with their SHA-256 printed. The Q4_K_M SHA-256 is
the snapshot's identity: it goes in the repo manifest next to the version
and the R2 URL (Phase 5).


In [ ]:
# 9 — Convert the merged model to one Q4_K_M GGUF (the snapshot).
# Approved build-time tool only (Gary, 2026-11-10). Two steps:
# F16 convert first (q4_k_m is NOT a valid --outtype), then llama-quantize.
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
!pip install -q -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py /content/drive/MyDrive/llmbuilder-lora/run1-merged \
  --outfile /content/drive/MyDrive/llmbuilder-lora/lucid-1b-lora-F16.gguf
!ls -lh /content/drive/MyDrive/llmbuilder-lora/lucid-1b-lora-F16.gguf
!sha256sum /content/drive/MyDrive/llmbuilder-lora/lucid-1b-lora-F16.gguf
!cmake -S llama.cpp -B llama.cpp/build -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build llama.cpp/build --target llama-quantize -j
!llama.cpp/build/bin/llama-quantize /content/drive/MyDrive/llmbuilder-lora/lucid-1b-lora-F16.gguf /content/drive/MyDrive/llmbuilder-lora/lucid-1b-lora-Q4_K_M.gguf Q4_K_M
!ls -lh /content/drive/MyDrive/llmbuilder-lora/lucid-1b-lora-Q4_K_M.gguf
!sha256sum /content/drive/MyDrive/llmbuilder-lora/lucid-1b-lora-Q4_K_M.gguf


In [ ]:
# 10 — Verify everything that should exist, then download the GGUF.
import hashlib
from pathlib import Path

OUT = Path("/content/drive/MyDrive/llmbuilder-lora")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size:>12,}  {p.relative_to(OUT)}")

for name in ("lucid-1b-lora-F16.gguf", "lucid-1b-lora-Q4_K_M.gguf"):
    gguf = OUT / name
    if gguf.exists():
        h = hashlib.sha256()
        with gguf.open("rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                h.update(chunk)
        print(f"{name} SHA-256:", h.hexdigest())
    else:
        print(f"No {name} yet.")

gguf = OUT / "lucid-1b-lora-Q4_K_M.gguf"
if gguf.exists():
    from google.colab import files
    files.download(str(gguf))   # pulls the GGUF to the machine running the browser
else:
    print("No Q4_K_M GGUF yet — run step 9 (two-step F16 -> Q4_K_M).")
    print("Adapter + merged model are on Drive and safe.")


## What happens next (NOT in this notebook)

- **Phase 4 — Evaluate + gate:** benchmark the fine-tune on the 168-task
  suite, check all 10 protected floors, A/B vs the stock 1B. No promotion
  without a measured improvement and no floor breach. (Separate work —
  the gate runner is `run_full_candidate_gate.py`; the stock-1B baseline
  script is Lucid-side, to be written.)
- **Phase 5 — Snapshot distribution:** upload the promoted GGUF to R2,
  publish the manifest (version + SHA-256 + URL). The SHA-256 printed
  above is the snapshot's identity.
- **Phase 6–7:** built-in inference engine + chat wiring, then ship to
  Gary's Mac first and verify live. **Then the starship.**

This notebook is the stable GPU link. It will eventually supply a
start-stop event pool: GPU sessions started and stopped on demand for
designated jobs (the GPU pool), using the same per-call progress pass
the ping server already serves. That pool is a separate build, not this
notebook — this notebook stays the single stable training/convert path.
